In [0]:
##Read data from Azure Data Lake

import urllib.request

url = "https://bb49.s3.ap-south-1.amazonaws.com/data.txt"

user = spark.sql("SELECT current_user()").collect()[0][0]

path = f"/Workspace/Users/{user}/data.txt"

urllib.request.urlretrieve(url, path)

df = spark.read.option("header", "true").option("inferSchema", "true").parquet("file:" + path)


df.show()
df.printSchema()





In [0]:

##Transforming the data - collecting the score for each username

from pyspark.sql.functions import *

aggdf = df.groupBy("username")\
    .agg(collect_list("score"))\
        .withColumnRenamed("collect_list(score)", "scores")

aggdf.show()

aggdf.printSchema()


# aggdf.select(
#     "username",
#     slice(col("collect_list(score)"), 1, 10).alias("scores_preview"),  # first 10 elements
#     size(col("collect_list(score)")).alias("total_count")               # so you know how many were cut off
# ).show(truncate=False)

In [0]:
%sql

-- Creating a lakehouse db using sql


create database if not exists zeyodb




In [0]:
# Writing the transformed data to lakehouse table using pyspark

aggdf.write.mode("overwrite")\
    .saveAsTable("zeyodb.adl_tab")


In [0]:
%sql

--validating the lakehouse table write

select * from zeyodb.adl_tab limit 20